## 📦 Setup & Utilities
This cell defines utility functions for running shell commands and setting the working directory. Make sure to run it before executing any other steps in the notebook.

In [1]:
# 📦 Imports
import subprocess
import shlex
import os

def run_command(command):
    """
    Executes a command in the shell and allows it to print directly to the notebook.
    """
    try:
        project_root = os.getcwd()
        if os.name == 'nt' and command.startswith('python3'):
            command = 'python' + command[len('python3'):]
        print(f"Executing command:\n{command}\n")
        subprocess.run(
            shlex.split(command), 
            check=True,
            cwd=project_root
        )
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## ✅ Install or Update Requirements
Use this step to install all required Python packages from requirements.txt. Run this only once or whenever dependencies change.

In [5]:
!python3 -m pip install git+https://github.com/lucasb-eyer/pydensecrf.git

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Cloning https://github.com/lucasb-eyer/pydensecrf.git to /tmp/pip-req-build-_kvq074_
  Running command git clone --filter=blob:none --quiet https://github.com/lucasb-eyer/pydensecrf.git /tmp/pip-req-build-_kvq074_
  Resolved https://github.com/lucasb-eyer/pydensecrf.git to commit 2723c7fa4f2ead16ae1ce3d8afe977724bb8f87f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pydensecrf: filename=pydensecrf-1.0-cp310-cp310-linux_x86_64.whl size=3405254 sha256=0f69487921ef5f34921ad4afc7c8b909c7b27aefcd500e7d4caf32394a801fd0
  Stored in directory: /tmp/pip-ephem-wheel-cache-0v4u9e7p/wheels/01/5b/61/87443ed3bf03dd2940375cf2f8b6fba88efece935465e490b0
Successfully built pydensecrf

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[n

In [6]:
# ✅ Install/Update Requirements
def install_requirements():
    print("\n--- Installing/Updating Requirements ---")
    command = "python3 -m pip install -r requirements.txt"
    run_command(command)
    print("--- Requirements Installation Finished ---")

# Run this cell when needed
install_requirements()


--- Installing/Updating Requirements ---
Executing command:
python3 -m pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
--- Requirements Installation Finished ---



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 🧹 Prepare the Dataset
This cell runs the data preparation script. It typically includes dataset splitting, formatting, or any pre-processing needed before training.

In [75]:
# 🧹 Prepare Data
def prepare_data():
    print("\n--- Preparing Data ---")
    command = "python3 -m utils.prepare_data"
    run_command(command)
    print("--- Data Preparation Finished ---")

# Run this cell when needed
prepare_data()


--- Preparing Data ---
Executing command:
python3 -m utils.prepare_data



Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/omniflow/test_nwdi/NN_Project_2025/utils/prepare_data.py", line 205, in <module>
    train_loader, val_loader, test_loader = get_data_loaders(batch_size=8, use_ndwi=True, compute_stats=COMPUTE_NDWI_STATS)
  File "/home/omniflow/test_nwdi/NN_Project_2025/utils/prepare_data.py", line 109, in get_data_loaders
    train_dataset = WaterBodiesDataset(
TypeError: WaterBodiesDataset.__init__() got an unexpected keyword argument 'transform'


Loading dataset paths...
Found 2841 images and 2841 masks

Computing NDWI statistics from sample data...
Computing NDWI stats from 50 samples...
Successfully processed 50/50 samples
NDWI Statistics:
  Mean: 0.059
  Std: 0.173
  Range: [-0.613, 0.746]
  Percentiles: {'25': -0.06422018259763718, '50': 0.0, '75': 0.17475727200508118, '90': 0.28104573488235474, '95': 0.3380281627178192}
Using NDWI threshold: 0.146

Splitting dataset...
Dataset split:
  Train: 2272 samples
  Validation: 284 samples
  Test: 285 samples
An unexpected error occurred: Command '['python3', '-m', 'utils.prepare_data']' returned non-zero exit status 1.
--- Data Preparation Finished ---


## 👁️ Visualize Dataset Samples
Use this cell to visualize a batch of images and masks from the dataset. Helpful to verify that the preprocessing step worked correctly.

In [57]:
# 👁️ Visualize Dataset
def visualize_dataset():
    print("\n--- Visualizing Dataset ---")
    command = "python3 utils/visualize_dataset.py"
    run_command(command)
    print("--- Dataset Visualization Finished ---")

# Run this cell when needed
visualize_dataset()


--- Visualizing Dataset ---
Executing command:
python3 utils/visualize_dataset.py

--- Visualizing Dataset ---
✅ Saved visualization to: outputs/visualized_sample.png
--- Dataset Visualization Finished ---


## 🧠 Train a Model
This is the interactive section for model training. You can choose from different architectures and set training hyperparameters. Default values are provided for ease of use.

In [78]:
# 🧠 Train Model
def train_model():
    print("\n--- Train a New Model ---")

    model_map = {
        "1": "aer-unet",
        "2": "unet++-pretrained-encoder",
        "3": "segformer-b4",
    }

    print("1. AER U-Net")
    print("2. U-Net++ (Pre-trained Encoder)")
    print("3. SegFormer-B4")

    model_choice = input("Enter your choice (1-3): ")
    model = model_map.get(model_choice)
    if not model:
        print("Invalid model selection.")
        return

    num_epochs = input("Number of epochs (default: 100): ") or "100"
    default_lr = "5e-5" if model == "segformer-b4" else "1e-4"
    learning_rate = input(f"Learning rate (default: {default_lr}): ") or default_lr
    batch_size = input("Batch size (default: 16): ") or "16"

    loss_options = "bce, dice, combined, focal, tversky, focal_lovasz"
    loss_type = input(f"Loss type ({loss_options}; default: focal_lovasz): ") or "focal_lovasz"

    command = f"python3 train.py --model {model} --num_epochs {num_epochs} --learning_rate {learning_rate} --batch_size {batch_size} --loss_type {loss_type}"

    if loss_type == "focal_lovasz":
        focal_weight = input("Focal loss weight (default: 0.5): ") or "0.5"
        lovasz_weight = input("Lovasz loss weight (default: 0.5): ") or "0.5"
        command += f" --focal_weight {focal_weight} --lovasz_weight {lovasz_weight}"

    if input("Use AMP? (Y/n): ").lower() != "n":
        command += " --use_amp"

    if model == "unet++-pretrained-encoder":
        if input("Use Deep Supervision? (Y/n): ").lower() != "n":
            command += " --deep_supervision"

    if input("Use Early Stopping? (Y/n): ").lower() != "n":
        command += " --early_stopping"

    if input("Use Gradient Clipping? (y/N): ").lower() == "y":
        command += " --gradient_clipping"
        
    if input("Use NDWI preprocessing AND as additional input channel? (y/N): ").lower() == "y":
        command += " --use_ndwi_as_input"

    if input("Disable NDWI completely (y/N): ").lower() == "y":
        command += " --no_ndwi"

    # if input("NDWI statistics during training (y/N): ").lower() == "y":
    #     command += " --show_ndwi_stats"

    print(f"\nFinal command:\n{command}\n")
    run_command(command)
    print("--- Model Training Finished ---")

# Run this cell to start training
train_model()


--- Train a New Model ---
1. AER U-Net
2. U-Net++ (Pre-trained Encoder)
3. SegFormer-B4


Enter your choice (1-3):  2
Number of epochs (default: 100):  
Learning rate (default: 1e-4):  
Batch size (default: 16):  
Loss type (bce, dice, combined, focal, tversky, focal_lovasz; default: focal_lovasz):  
Focal loss weight (default: 0.5):  
Lovasz loss weight (default: 0.5):  
Use AMP? (Y/n):  y
Use Deep Supervision? (Y/n):  y
Use Early Stopping? (Y/n):  y
Use Gradient Clipping? (y/N):  y
Use NDWI preprocessing AND as additional input channel? (y/N):  y
Disable NDWI completely (y/N):  n



Final command:
python3 train.py --model unet++-pretrained-encoder --num_epochs 100 --learning_rate 1e-4 --batch_size 16 --loss_type focal_lovasz --focal_weight 0.5 --lovasz_weight 0.5 --use_amp --deep_supervision --early_stopping --gradient_clipping --use_ndwi_as_input

Executing command:
python3 train.py --model unet++-pretrained-encoder --num_epochs 100 --learning_rate 1e-4 --batch_size 16 --loss_type focal_lovasz --focal_weight 0.5 --lovasz_weight 0.5 --use_amp --deep_supervision --early_stopping --gradient_clipping --use_ndwi_as_input



2025-06-23 16:38:23.449413: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-23 16:38:23.464257: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750696703.481242  460871 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750696703.486397  460871 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 16:38:23.503353: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Loading dataset paths...
Found 2841 images and 2841 masks

Computing NDWI statistics from sample data...
Computing NDWI stats from 50 samples...
Successfully processed 50/50 samples
NDWI Statistics:
  Mean: 0.027
  Std: 0.165
  Range: [-0.615, 0.710]
  Percentiles: {'25': -0.06719370186328888, '50': -0.010600706562399864, '75': 0.06206895783543587, '90': 0.26829269528388977, '95': 0.40384620428085327}
Using NDWI threshold: 0.109

Splitting dataset...
Dataset split:
  Train: 2272 samples
  Validation: 284 samples
  Test: 285 samples
Datasets created successfully!
Data loaders created with 2 workers and batch size 16

Testing data loaders...
Train Batch 1: Image torch.Size([16, 3, 256, 256]), Mask torch.Size([16, 1, 256, 256]), NDWI torch.Size([16, 1, 256, 256])
  Image range: [0.000, 0.776]
  Mask unique values: [0.0, 1.0]
  Water pixels (mask=1): 307503/1048576

Data loading complete!
Using device: cuda
Using NDWI as additional input channel. Total input channels: 4
Initializing U-Net+

...

Epoch 70/100 -> Train IoU: 0.9377, Val IoU: 0.8128, LR: 1.25e-05


Validation Epoch 71: 100%|██████████| 18/18 [00:03<00:00,  5.32it/s, Loss=0.1921, IoU=0.8468]


Epoch 71/100 -> Train IoU: 0.9373, Val IoU: 0.8118, LR: 1.25e-05


Validation Epoch 72: 100%|██████████| 18/18 [00:03<00:00,  5.42it/s, Loss=0.1869, IoU=0.8514]


Epoch 72/100 -> Train IoU: 0.9363, Val IoU: 0.8119, LR: 1.25e-05
Early stopping triggered at epoch 72. Best Val IoU: 0.8150

Training history and plots saved to: checkpoints/unet++-pretrained-encoder_20250623-163826
--- Model Training Finished ---


## 🧪 Evaluate the Trained Model
This will launch the evaluation script. You can use it to interactively test the trained model on validation or test sets.

In [92]:
# 🧪 Evaluate Model
def evaluate_model():
    print("\n--- Evaluating Model ---")
    model_checkpoint = "/home/omniflow/test_nwdi/NN_Project_2025/checkpoints/unet++-pretrained-encoder_20250623-163826/best.pth"  
    command = f"python3 evaluate.py --path {model_checkpoint}"
    run_command(command)
    print("--- Evaluation Finished ---")
    
# Run this cell when needed
evaluate_model()


--- Evaluating Model ---
Executing command:
python3 evaluate.py --path /home/omniflow/test_nwdi/NN_Project_2025/checkpoints/unet++-pretrained-encoder_20250623-163826/best.pth --use_tta



2025-06-23 17:38:33.550819: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-23 17:38:33.565758: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750700313.582157  465166 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750700313.587421  465166 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 17:38:33.604452: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

An unexpected error occurred: Command '['python3', 'evaluate.py', '--path', '/home/omniflow/test_nwdi/NN_Project_2025/checkpoints/unet++-pretrained-encoder_20250623-163826/best.pth', '--use_tta']' returned non-zero exit status 2.
--- Evaluation Finished ---
